In [1]:
import pandas as pd

df=pd.read_csv(r"student_dataset_ml.csv")
print(df.shape)
print(df.head)

(5000, 27)
<bound method NDFrame.head of       Student_ID  CGPA  Attendance  Backlogs  Year  Python  Java  DSA  SQL  \
0              1  6.40          66         1     3       0     0    1    0   
1              2  6.35          83         3     4       0     0    0    1   
2              3  6.80          81         2     4       2     2    2    2   
3              4  9.17          90         0     3       0     1    0    1   
4              5  8.09          83         1     4       1     2    1    0   
...          ...   ...         ...       ...   ...     ...   ...  ...  ...   
4995        4996  8.23          83         1     4       1     1    1    1   
4996        4997  8.46          89         1     2       2     1    2    2   
4997        4998  7.55          69         1     4       2     2    2    1   
4998        4999  5.62          78         2     4       0     0    1    0   
4999        5000  7.32          90         2     4       2     1    1    1   

      OOP  ...  Projec

In [2]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate Student IDs:")
print(df["Student_ID"].duplicated().sum())

print("\nTarget distribution:")
print(df["EmployabilityStatus"].value_counts().sort_index())

print("\nData types:")
print(df.dtypes)

Shape: (5000, 27)

Missing values:
Student_ID             0
CGPA                   0
Attendance             0
Backlogs               0
Year                   0
Python                 0
Java                   0
DSA                    0
SQL                    0
OOP                    0
DBMS                   0
OS                     0
CN                     0
Git                    0
Linux                  0
Docker                 0
Cloud                  0
Projects               0
Internship             0
Certifications         0
Hackathons             0
OpenSource             0
LeetCode               0
Communication          0
Teamwork               0
Leadership             0
EmployabilityStatus    0
dtype: int64

Duplicate Student IDs:
0

Target distribution:
EmployabilityStatus
0    2287
1    1518
2    1195
Name: count, dtype: int64

Data types:
Student_ID               int64
CGPA                   float64
Attendance               int64
Backlogs                 int64
Year            

In [3]:
# ============================================================
# STEP 4: CREATE SYNTHETIC TEMPORAL FEATURE DATA
# ============================================================

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Three feature snapshots
snapshot_dates = [
    pd.Timestamp("2026-01-01", tz="UTC"),
    pd.Timestamp("2026-04-01", tz="UTC"),
    pd.Timestamp("2026-07-01", tz="UTC")
]

temporal_rows = []

for _, student in df.iterrows():

    for snapshot_index, timestamp in enumerate(snapshot_dates):

        row = student.copy()

        # ----------------------------------------------------
        # January: simulate an earlier student state
        # ----------------------------------------------------
        if snapshot_index == 0:

            row["CGPA"] = max(
                0,
                row["CGPA"] - rng.uniform(0.3, 0.8)
            )

            row["Attendance"] = max(
                0,
                row["Attendance"] - rng.integers(5, 15)
            )

            row["Projects"] = max(
                0,
                row["Projects"] - rng.integers(1, 3)
            )

            row["LeetCode"] = max(
                0,
                row["LeetCode"] - rng.integers(30, 150)
            )

        # ----------------------------------------------------
        # April: simulate an intermediate student state
        # ----------------------------------------------------
        elif snapshot_index == 1:

            row["CGPA"] = max(
                0,
                row["CGPA"] - rng.uniform(0.1, 0.4)
            )

            row["Attendance"] = max(
                0,
                row["Attendance"] - rng.integers(2, 8)
            )

            row["Projects"] = max(
                0,
                row["Projects"] - rng.integers(0, 1)
            )

            row["LeetCode"] = max(
                0,
                row["LeetCode"] - rng.integers(10, 70)
            )

        # ----------------------------------------------------
        # July: use the original generated values
        # ----------------------------------------------------

        row["event_timestamp"] = timestamp

        temporal_rows.append(row)


# Create DataFrame
temporal_df = pd.DataFrame(temporal_rows)

# Put ID and timestamp first
ordered_columns = (
    ["Student_ID", "event_timestamp"]
    + [
        c for c in temporal_df.columns
        if c not in ["Student_ID", "event_timestamp"]
    ]
)

temporal_df = temporal_df[ordered_columns]

print("Original dataset shape :", df.shape)
print("Temporal dataset shape:", temporal_df.shape)

display(
    temporal_df[
        [
            "Student_ID",
            "event_timestamp",
            "CGPA",
            "Attendance",
            "Projects",
            "LeetCode",
            "EmployabilityStatus"
        ]
    ].head(12)
)

Original dataset shape : (5000, 27)
Temporal dataset shape: (15000, 28)


,Student_ID,event_timestamp,CGPA,Attendance,Projects,LeetCode,EmployabilityStatus
0,1.0,2026-01-01 00:00:00+00:00,5.713022,55.0,1.0,0.0,2.0
0,1.0,2026-04-01 00:00:00+00:00,6.090790,59.0,2.0,30.0,2.0
0,1.0,2026-07-01 00:00:00+00:00,6.400000,66.0,2.0,52.0,2.0
1,2.0,2026-01-01 00:00:00+00:00,5.562189,78.0,0.0,0.0,2.0
1,2.0,2026-04-01 00:00:00+00:00,6.014181,78.0,2.0,12.0,2.0
1,2.0,2026-07-01 00:00:00+00:00,6.350000,83.0,2.0,29.0,2.0
2,3.0,2026-01-01 00:00:00+00:00,6.274807,71.0,5.0,457.0,1.0
2,3.0,2026-04-01 00:00:00+00:00,6.506840,74.0,6.0,474.0,1.0
2,3.0,2026-07-01 00:00:00+00:00,6.800000,81.0,6.0,508.0,1.0
3,4.0,2026-01-01 00:00:00+00:00,8.648293,77.0,0.0,0.0,2.0


In [4]:
# ============================================================
# STEP 5: INSPECT ONE STUDENT'S HISTORY
# ============================================================

student_id = 1

student_history = (
    temporal_df[
        temporal_df["Student_ID"] == student_id
    ]
    .sort_values("event_timestamp")
)

display(
    student_history[
        [
            "Student_ID",
            "event_timestamp",
            "CGPA",
            "Attendance",
            "Projects",
            "LeetCode",
            "EmployabilityStatus"
        ]
    ]
)

,Student_ID,event_timestamp,CGPA,Attendance,Projects,LeetCode,EmployabilityStatus
0,1.0,2026-01-01 00:00:00+00:00,5.713022,55.0,1.0,0.0,2.0
0,1.0,2026-04-01 00:00:00+00:00,6.090790,59.0,2.0,30.0,2.0
0,1.0,2026-07-01 00:00:00+00:00,6.400000,66.0,2.0,52.0,2.0


In [5]:
# ============================================================
# STEP 6: SAVE TEMPORAL DATA FOR FEAST
# ============================================================

from pathlib import Path

# Create data directory if it doesn't exist
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

TEMPORAL_FILE = data_dir / "student_features.parquet"

temporal_df.to_parquet(
    TEMPORAL_FILE,
    index=False
)

print("Saved:", TEMPORAL_FILE)

# Verify
check = pd.read_parquet(TEMPORAL_FILE)

print("Saved dataset shape:", check.shape)
display(check.head())

Saved: data\student_features.parquet
Saved dataset shape: (15000, 28)


,Student_ID,event_timestamp,CGPA,Attendance,Backlogs,Year,Python,Java,DSA,SQL,...,Projects,Internship,Certifications,Hackathons,OpenSource,LeetCode,Communication,Teamwork,Leadership,EmployabilityStatus
0,1.0,2026-01-01 00:00:00+00:00,5.713022,55.0,1.0,3.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,2.0
1,1.0,2026-04-01 00:00:00+00:00,6.090790,59.0,1.0,3.0,0.0,0.0,1.0,0.0,...,2.0,0.0,1.0,1.0,0.0,30.0,1.0,0.0,1.0,2.0
2,1.0,2026-07-01 00:00:00+00:00,6.400000,66.0,1.0,3.0,0.0,0.0,1.0,0.0,...,2.0,0.0,1.0,1.0,0.0,52.0,1.0,0.0,1.0,2.0
3,2.0,2026-01-01 00:00:00+00:00,5.562189,78.0,3.0,4.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0
4,2.0,2026-04-01 00:00:00+00:00,6.014181,78.0,3.0,4.0,0.0,0.0,0.0,1.0,...,2.0,0.0,0.0,1.0,0.0,12.0,0.0,0.0,1.0,2.0


In [6]:
from pathlib import Path
import shutil

# Current notebook directory
current_dir = Path.cwd()

# The file we accidentally created
wrong_file = current_dir / "data" / "student_features.parquet"

# Correct location: parent project folder / data
project_dir = current_dir.parent
correct_data_dir = project_dir / "data"
correct_data_dir.mkdir(parents=True, exist_ok=True)

correct_file = correct_data_dir / "student_features.parquet"

# Move the file
if wrong_file.exists():
    shutil.move(str(wrong_file), str(correct_file))

print("Project directory:")
print(project_dir)

print("\nCorrect Parquet location:")
print(correct_file)

print("\nExists:", correct_file.exists())

Project directory:
C:\Users\Gowtham\student_feast_project

Correct Parquet location:
C:\Users\Gowtham\student_feast_project\data\student_features.parquet

Exists: True


In [7]:
from pathlib import Path

project_dir = Path.cwd().parent

print("PROJECT:")
print(project_dir)

print("\nFiles/folders:")
for item in project_dir.iterdir():
    print(" ", item.name)

PROJECT:
C:\Users\Gowtham\student_feast_project

Files/folders:
  data
  feature_store.yaml


In [8]:
parquet_path = Path.cwd().parent / "data" / "student_features.parquet"

check = pd.read_parquet(parquet_path)

print("Path:")
print(parquet_path)

print("\nShape:")
print(check.shape)

print("\nColumns:")
print(check.columns.tolist())

Path:
C:\Users\Gowtham\student_feast_project\data\student_features.parquet

Shape:
(15000, 28)

Columns:
['Student_ID', 'event_timestamp', 'CGPA', 'Attendance', 'Backlogs', 'Year', 'Python', 'Java', 'DSA', 'SQL', 'OOP', 'DBMS', 'OS', 'CN', 'Git', 'Linux', 'Docker', 'Cloud', 'Projects', 'Internship', 'Certifications', 'Hackathons', 'OpenSource', 'LeetCode', 'Communication', 'Teamwork', 'Leadership', 'EmployabilityStatus']


In [9]:
# ============================================================
# STEP 7: CREATE FEAST CONFIGURATION
# ============================================================

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"

print("Feast project directory:")
print(PROJECT_DIR)

print("\nFeature data:")
print(DATA_DIR / "student_features.parquet")

Feast project directory:
C:\Users\Gowtham\student_feast_project

Feature data:
C:\Users\Gowtham\student_feast_project\data\student_features.parquet


In [10]:
# Create feature_store.yaml

yaml_content = f"""
project: student_employability

registry: {PROJECT_DIR / "data" / "registry.db"}

provider: local

online_store:
    type: sqlite
    path: {PROJECT_DIR / "data" / "online_store.db"}
"""

yaml_path = PROJECT_DIR / "feature_store.yaml"

yaml_path.write_text(
    yaml_content,
    encoding="utf-8"
)

print("Created:")
print(yaml_path)

print("\nContents:")
print(yaml_path.read_text())

Created:
C:\Users\Gowtham\student_feast_project\feature_store.yaml

Contents:

project: student_employability

registry: C:\Users\Gowtham\student_feast_project\data\registry.db

provider: local

online_store:
    type: sqlite
    path: C:\Users\Gowtham\student_feast_project\data\online_store.db



In [11]:
from feast import Entity, ValueType

student_entity = Entity(
    name="student",
    join_keys=["Student_ID"],
    value_type=ValueType.INT64,
    description="Student identified by Student_ID"
)

print("Entity created successfully.")
print(student_entity)

Entity created successfully.
{
  "spec": {
    "name": "student",
    "valueType": "INT64",
    "description": "Student identified by Student_ID",
    "joinKey": "Student_ID"
  },
  "meta": {}
}


In [12]:
from pathlib import Path
from feast import FileSource

# Feast project directory
PROJECT_DIR = Path(r"C:\Users\Gowtham\student_feast_project")

# Data directory
DATA_DIR = PROJECT_DIR / "data"

# Parquet file
PARQUET_PATH = DATA_DIR / "student_features.parquet"

print("Parquet exists:", PARQUET_PATH.exists())
print("Path:", PARQUET_PATH)

student_source = FileSource(
    path=str(PARQUET_PATH),
    timestamp_field="event_timestamp"
)

print("\nFileSource created successfully.")

Parquet exists: True
Path: C:\Users\Gowtham\student_feast_project\data\student_features.parquet

FileSource created successfully.


In [13]:
# ============================================================
# STEP: FEATURE ENGINEERING
# ============================================================

technical_skill_columns = [
    "Python",
    "Java",
    "DSA",
    "SQL",
    "OOP",
    "DBMS",
    "OS",
    "CN",
    "Git",
    "Linux",
    "Docker",
    "Cloud"
]

temporal_df["TechnicalSkillScore"] = (
    temporal_df[technical_skill_columns].mean(axis=1) / 2 * 100
)

print("TechnicalSkillScore created.")

display(
    temporal_df[
        [
            "Student_ID",
            "Python",
            "Java",
            "DSA",
            "SQL",
            "TechnicalSkillScore"
        ]
    ].head(10)
)

TechnicalSkillScore created.


,Student_ID,Python,Java,DSA,SQL,TechnicalSkillScore
0,1.0,0.0,0.0,1.0,0.0,8.333333
0,1.0,0.0,0.0,1.0,0.0,8.333333
0,1.0,0.0,0.0,1.0,0.0,8.333333
1,2.0,0.0,0.0,0.0,1.0,8.333333
1,2.0,0.0,0.0,0.0,1.0,8.333333
1,2.0,0.0,0.0,0.0,1.0,8.333333
2,3.0,2.0,2.0,2.0,2.0,66.666667
2,3.0,2.0,2.0,2.0,2.0,66.666667
2,3.0,2.0,2.0,2.0,2.0,66.666667
3,4.0,0.0,1.0,0.0,1.0,37.500000


In [14]:
# ============================================================
# STEP: CREATE THE FEAST FEATURE VIEW
# ============================================================

from datetime import timedelta

from feast import FeatureView, Field
from feast.types import Int64, Float64

student_features = FeatureView(
    name="student_features",

    # Which entity these features belong to
    entities=[student_entity],

    # How long a feature value remains valid
    ttl=timedelta(days=365),

    # Our 26 ML features
    schema=[
        Field(name="CGPA", dtype=Float64),
        Field(name="Attendance", dtype=Int64),
        Field(name="Backlogs", dtype=Int64),
        Field(name="Year", dtype=Int64),

        Field(name="Python", dtype=Int64),
        Field(name="Java", dtype=Int64),
        Field(name="DSA", dtype=Int64),
        Field(name="SQL", dtype=Int64),
        Field(name="OOP", dtype=Int64),
        Field(name="DBMS", dtype=Int64),
        Field(name="OS", dtype=Int64),
        Field(name="CN", dtype=Int64),
        Field(name="Git", dtype=Int64),
        Field(name="Linux", dtype=Int64),
        Field(name="Docker", dtype=Int64),
        Field(name="Cloud", dtype=Int64),
        Field(name="TechnicalSkillScore", dtype=Float64),
        Field(name="Projects", dtype=Int64),
        Field(name="Internship", dtype=Int64),
        Field(name="Certifications", dtype=Int64),
        Field(name="Hackathons", dtype=Int64),
        Field(name="OpenSource", dtype=Int64),
        Field(name="LeetCode", dtype=Int64),

        Field(name="Communication", dtype=Int64),
        Field(name="Teamwork", dtype=Int64),
        Field(name="Leadership", dtype=Int64),
    ],

    # Where Feast gets the historical data
    source=student_source,

    # Make features available for online serving later
    online=True,
)

print("FeatureView created successfully.")
print(student_features)

FeatureView created successfully.
{
  "spec": {
    "name": "student_features",
    "entities": [
      "student"
    ],
    "features": [
      {
        "name": "CGPA",
        "valueType": "DOUBLE"
      },
      {
        "name": "Attendance",
        "valueType": "INT64"
      },
      {
        "name": "Backlogs",
        "valueType": "INT64"
      },
      {
        "name": "Year",
        "valueType": "INT64"
      },
      {
        "name": "Python",
        "valueType": "INT64"
      },
      {
        "name": "Java",
        "valueType": "INT64"
      },
      {
        "name": "DSA",
        "valueType": "INT64"
      },
      {
        "name": "SQL",
        "valueType": "INT64"
      },
      {
        "name": "OOP",
        "valueType": "INT64"
      },
      {
        "name": "DBMS",
        "valueType": "INT64"
      },
      {
        "name": "OS",
        "valueType": "INT64"
      },
      {
        "name": "CN",
        "valueType": "INT64"
      },
      {
       

In [15]:
# ============================================================
# STEP: INITIALIZE FEAST FEATURE STORE
# ============================================================

from feast import FeatureStore

store = FeatureStore(
    repo_path=str(PROJECT_DIR)
)

print("FeatureStore initialized successfully.")

FeatureStore initialized successfully.


In [16]:
# ============================================================
# STEP: REGISTER ENTITY + FEATURE VIEW WITH FEAST
# ============================================================

store.apply([
    student_entity,
    student_features
])

print("Feast objects registered successfully.")

Feast objects registered successfully.


In [17]:
# ============================================================
# STEP: CREATE ENTITY DATAFRAME FOR HISTORICAL RETRIEVAL
# ============================================================

import pandas as pd

# Load our temporal dataset
temporal_data = pd.read_parquet(
    PARQUET_PATH
)

# Entity dataframe
entity_df = temporal_data[
    [
        "Student_ID",
        "event_timestamp",
        "EmployabilityStatus"
    ]
].copy()

print("Entity dataframe shape:")
print(entity_df.shape)

print("\nEntity dataframe:")
display(entity_df.head(10))

Entity dataframe shape:
(15000, 3)

Entity dataframe:


,Student_ID,event_timestamp,EmployabilityStatus
0,1.0,2026-01-01 00:00:00+00:00,2.0
1,1.0,2026-04-01 00:00:00+00:00,2.0
2,1.0,2026-07-01 00:00:00+00:00,2.0
3,2.0,2026-01-01 00:00:00+00:00,2.0
4,2.0,2026-04-01 00:00:00+00:00,2.0
5,2.0,2026-07-01 00:00:00+00:00,2.0
6,3.0,2026-01-01 00:00:00+00:00,1.0
7,3.0,2026-04-01 00:00:00+00:00,1.0
8,3.0,2026-07-01 00:00:00+00:00,1.0
9,4.0,2026-01-01 00:00:00+00:00,2.0


In [20]:
# ============================================================
# FIX: ADD ENGINEERED FEATURE AND SAVE PARQUET
# ============================================================

technical_skill_columns = [
    "Python",
    "Java",
    "DSA",
    "SQL",
    "OOP",
    "DBMS",
    "OS",
    "CN",
    "Git",
    "Linux",
    "Docker",
    "Cloud"
]

# Add the engineered feature
temporal_df["TechnicalSkillScore"] = (
    temporal_df[technical_skill_columns].mean(axis=1) / 2 * 100
)

print("Temporal dataframe shape:", temporal_df.shape)
print("TechnicalSkillScore exists:",
      "TechnicalSkillScore" in temporal_df.columns)

# Save UPDATED dataframe
temporal_df.to_parquet(
    PARQUET_PATH,
    index=False
)

print("\nUpdated Parquet saved.")

Temporal dataframe shape: (15000, 29)
TechnicalSkillScore exists: True

Updated Parquet saved.


In [21]:
# Check the actual Parquet file
check = pd.read_parquet(PARQUET_PATH)

print("Shape:", check.shape)
print("\nHas TechnicalSkillScore:",
      "TechnicalSkillScore" in check.columns)

print("\nColumns:")
print(check.columns.tolist())

Shape: (15000, 29)

Has TechnicalSkillScore: True

Columns:
['Student_ID', 'event_timestamp', 'CGPA', 'Attendance', 'Backlogs', 'Year', 'Python', 'Java', 'DSA', 'SQL', 'OOP', 'DBMS', 'OS', 'CN', 'Git', 'Linux', 'Docker', 'Cloud', 'Projects', 'Internship', 'Certifications', 'Hackathons', 'OpenSource', 'LeetCode', 'Communication', 'Teamwork', 'Leadership', 'EmployabilityStatus', 'TechnicalSkillScore']


In [22]:
# ============================================================
# STEP: HISTORICAL FEATURE RETRIEVAL
# ============================================================

feature_refs = [
    "student_features:CGPA",
    "student_features:Attendance",
    "student_features:Backlogs",
    "student_features:Year",

    "student_features:Python",
    "student_features:Java",
    "student_features:DSA",
    "student_features:SQL",
    "student_features:OOP",
    "student_features:DBMS",
    "student_features:OS",
    "student_features:CN",
    "student_features:Git",
    "student_features:Linux",
    "student_features:Docker",
    "student_features:Cloud",
    "student_features:TechnicalSkillScore",
    "student_features:Projects",
    "student_features:Internship",
    "student_features:Certifications",
    "student_features:Hackathons",
    "student_features:OpenSource",
    "student_features:LeetCode",

    "student_features:Communication",
    "student_features:Teamwork",
    "student_features:Leadership",
]

print("Starting historical feature retrieval...")

historical_job = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs
)

historical_df = historical_job.to_df()

print("\nHistorical retrieval completed.")

print("Shape:", historical_df.shape)

display(historical_df.head())

Starting historical feature retrieval...

Historical retrieval completed.
Shape: (15000, 29)


,Student_ID,event_timestamp,EmployabilityStatus,CGPA,Attendance,Backlogs,Year,Python,Java,DSA,...,TechnicalSkillScore,Projects,Internship,Certifications,Hackathons,OpenSource,LeetCode,Communication,Teamwork,Leadership
0,1.0,2026-01-01 00:00:00+00:00,2.0,5.713022,55.0,1.0,3.0,0.0,0.0,1.0,...,8.333333,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
1,2483.0,2026-01-01 00:00:00+00:00,2.0,5.476046,72.0,2.0,4.0,1.0,1.0,1.0,...,16.666667,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,4587.0,2026-01-01 00:00:00+00:00,1.0,6.997374,70.0,0.0,2.0,2.0,2.0,2.0,...,58.333333,3.0,1.0,4.0,5.0,1.0,659.0,2.0,1.0,2.0
3,2482.0,2026-01-01 00:00:00+00:00,1.0,8.135799,82.0,1.0,3.0,2.0,2.0,2.0,...,75.000000,3.0,0.0,3.0,2.0,0.0,552.0,1.0,2.0,0.0
4,2481.0,2026-01-01 00:00:00+00:00,1.0,9.215721,92.0,0.0,2.0,2.0,2.0,2.0,...,79.166667,3.0,1.0,3.0,2.0,1.0,473.0,1.0,1.0,2.0


In [23]:
# ============================================================
# STEP: VALIDATE FEAST HISTORICAL DATASET
# ============================================================

print("Shape:", historical_df.shape)

print("\nMissing values:")
print(historical_df.isnull().sum())

print("\nData types:")
print(historical_df.dtypes)

print("\nColumns:")
print(historical_df.columns.tolist())

Shape: (15000, 29)

Missing values:
Student_ID             0
event_timestamp        0
EmployabilityStatus    0
CGPA                   0
Attendance             0
Backlogs               0
Year                   0
Python                 0
Java                   0
DSA                    0
SQL                    0
OOP                    0
DBMS                   0
OS                     0
CN                     0
Git                    0
Linux                  0
Docker                 0
Cloud                  0
TechnicalSkillScore    0
Projects               0
Internship             0
Certifications         0
Hackathons             0
OpenSource             0
LeetCode               0
Communication          0
Teamwork               0
Leadership             0
dtype: int64

Data types:
Student_ID                         float64
event_timestamp        datetime64[ns, UTC]
EmployabilityStatus                float64
CGPA                               float64
Attendance                         float6

In [24]:
# ============================================================
# STEP: SAVE FEAST TRAINING DATASET
# ============================================================

FEAST_TRAINING_FILE = DATA_DIR / "feast_training_dataset.parquet"

historical_df.to_parquet(
    FEAST_TRAINING_FILE,
    index=False
)

print("Saved Feast training dataset:")
print(FEAST_TRAINING_FILE)

Saved Feast training dataset:
C:\Users\Gowtham\student_feast_project\data\feast_training_dataset.parquet


In [25]:
# ============================================================
# STEP: PREPARE X AND y
# ============================================================

TARGET = "EmployabilityStatus"

DROP_COLUMNS = [
    "Student_ID",
    "event_timestamp",
    TARGET
]

X = historical_df.drop(
    columns=DROP_COLUMNS
)

y = historical_df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts().sort_index())

X shape: (15000, 26)
y shape: (15000,)

Feature columns:
['CGPA', 'Attendance', 'Backlogs', 'Year', 'Python', 'Java', 'DSA', 'SQL', 'OOP', 'DBMS', 'OS', 'CN', 'Git', 'Linux', 'Docker', 'Cloud', 'TechnicalSkillScore', 'Projects', 'Internship', 'Certifications', 'Hackathons', 'OpenSource', 'LeetCode', 'Communication', 'Teamwork', 'Leadership']

Target distribution:
EmployabilityStatus
0.0    6861
1.0    4554
2.0    3585
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

feast_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

feast_model.fit(X_train, y_train)

y_pred = feast_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n====================================")
print("FEAST PIPELINE RESULTS")
print("====================================")

print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training samples: 12000
Testing samples : 3000

FEAST PIPELINE RESULTS
Accuracy: 0.924

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.91      0.92      1372
         1.0       0.94      0.92      0.93       911
         2.0       0.91      0.95      0.93       717

    accuracy                           0.92      3000
   macro avg       0.92      0.93      0.93      3000
weighted avg       0.92      0.92      0.92      3000


Confusion Matrix:
[[1248   53   71]
 [  70  841    0]
 [  32    2  683]]


In [27]:
# ============================================================
# STEP: SAVE FEAST-BASED MODEL
# ============================================================

import joblib

MODEL_PATH = DATA_DIR / "feast_random_forest.pkl"

joblib.dump(
    feast_model,
    MODEL_PATH
)

print("Model saved to:")
print(MODEL_PATH)

Model saved to:
C:\Users\Gowtham\student_feast_project\data\feast_random_forest.pkl


In [28]:
# ============================================================
# STEP: MATERIALIZE FEATURES INTO ONLINE STORE
# ============================================================

from datetime import datetime, timezone

start_date = datetime(
    2026, 1, 1,
    tzinfo=timezone.utc
)

end_date = datetime(
    2026, 7, 1,
    tzinfo=timezone.utc
)

print("Starting materialization...")

store.materialize(
    start_date,
    end_date
)

print("Materialization completed.")

Starting materialization...
Materializing 1 feature views from 2026-01-01 00:00:00+00:00 to 2026-07-01 00:00:00+00:00 into the sqlite online store.

student_features:
Materialization completed.


In [29]:
# ============================================================
# STEP: ONLINE FEATURE RETRIEVAL
# ============================================================

student_id = 1

feature_refs = [
    "student_features:CGPA",
    "student_features:Attendance",
    "student_features:Backlogs",
    "student_features:Year",
    "student_features:Python",
    "student_features:Java",
    "student_features:DSA",
    "student_features:SQL",
    "student_features:OOP",
    "student_features:DBMS",
    "student_features:OS",
    "student_features:CN",
    "student_features:Git",
    "student_features:Linux",
    "student_features:Docker",
    "student_features:Cloud",
    "student_features:TechnicalSkillScore",
    "student_features:Projects",
    "student_features:Internship",
    "student_features:Certifications",
    "student_features:Hackathons",
    "student_features:OpenSource",
    "student_features:LeetCode",
    "student_features:Communication",
    "student_features:Teamwork",
    "student_features:Leadership",
]

online_features = store.get_online_features(
    features=feature_refs,
    entity_rows=[
        {
            "Student_ID": student_id
        }
    ]
).to_dict()

print("Online features retrieved:")
print(online_features)

Online features retrieved:
{'Student_ID': [1], 'OS': [0], 'Git': [0], 'TechnicalSkillScore': [8.333333333333332], 'DSA': [1], 'CGPA': [6.4], 'Teamwork': [0], 'DBMS': [0], 'Certifications': [1], 'SQL': [0], 'Hackathons': [1], 'Projects': [2], 'CN': [0], 'Cloud': [0], 'LeetCode': [52], 'Docker': [0], 'Internship': [0], 'Communication': [1], 'OpenSource': [0], 'Attendance': [66], 'Backlogs': [1], 'Python': [0], 'Linux': [0], 'Java': [0], 'Year': [3], 'OOP': [1], 'Leadership': [1]}


In [30]:
# ============================================================
# STEP: PREDICT USING ONLINE FEAST FEATURES
# ============================================================

feature_names = [
    "CGPA",
    "Attendance",
    "Backlogs",
    "Year",
    "Python",
    "Java",
    "DSA",
    "SQL",
    "OOP",
    "DBMS",
    "OS",
    "CN",
    "Git",
    "Linux",
    "Docker",
    "Cloud",
    "TechnicalSkillScore",
    "Projects",
    "Internship",
    "Certifications",
    "Hackathons",
    "OpenSource",
    "LeetCode",
    "Communication",
    "Teamwork",
    "Leadership"
]

# Convert Feast dictionary into DataFrame
X_online = pd.DataFrame({
    feature: online_features[feature]
    for feature in feature_names
})

print("Online feature vector:")
display(X_online)

# Predict
prediction = feast_model.predict(X_online)[0]

print("\n====================================")
print("EMPLOYABILITY PREDICTION")
print("====================================")
print("Student ID:", student_id)
print("Predicted class:", int(prediction))

Online feature vector:


,CGPA,Attendance,Backlogs,Year,Python,Java,DSA,SQL,OOP,DBMS,...,TechnicalSkillScore,Projects,Internship,Certifications,Hackathons,OpenSource,LeetCode,Communication,Teamwork,Leadership
0,6.4,66,1,3,0,0,1,0,1,0,...,8.333333,2,0,1,1,0,52,1,0,1



EMPLOYABILITY PREDICTION
Student ID: 1
Predicted class: 2


In [31]:
# ============================================================
# STEP: PREDICTION PROBABILITIES
# ============================================================

probabilities = feast_model.predict_proba(X_online)[0]

print("Prediction probabilities:")

for class_id, probability in zip(
    feast_model.classes_,
    probabilities
):
    print(
        f"Class {int(class_id)}: "
        f"{probability:.2%}"
    )

Prediction probabilities:
Class 0: 1.50%
Class 1: 0.00%
Class 2: 98.50%


In [32]:
# ============================================================
# STEP: SAVE UPDATED FEATURE DATASET
# ============================================================

temporal_df.to_parquet(
    PARQUET_PATH,
    index=False
)

print("Updated feature dataset saved.")
print("Shape:", temporal_df.shape)

Updated feature dataset saved.
Shape: (15000, 29)


In [34]:
# ============================================================
# COMPLETE FEAST ASSIGNMENT VERIFICATION - FEAST 0.65
# ============================================================

import pandas as pd

print("=" * 70)
print("       CURRICULUM-INDUSTRY SKILL GAP — FEAST VERIFICATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. ORIGINAL DATASET
# ------------------------------------------------------------

print("\n[1/8] ORIGINAL DATASET")

print("Shape:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate Student IDs:",
      int(df["Student_ID"].duplicated().sum()))

assert df.shape == (5000, 27)
assert df.isnull().sum().sum() == 0
assert df["Student_ID"].duplicated().sum() == 0

print("✓ Original dataset is valid")


# ------------------------------------------------------------
# 2. FEATURE DATASET
# ------------------------------------------------------------

print("\n[2/8] FEATURE DATASET")

feature_check = pd.read_parquet(PARQUET_PATH)

print("Shape:", feature_check.shape)

required_columns = [
    "Student_ID",
    "event_timestamp",

    "CGPA",
    "Attendance",
    "Backlogs",
    "Year",
    "Python",
    "Java",
    "DSA",
    "SQL",
    "OOP",
    "DBMS",
    "OS",
    "CN",
    "Git",
    "Linux",
    "Docker",
    "Cloud",
    "TechnicalSkillScore",
    "Projects",
    "Internship",
    "Certifications",
    "Hackathons",
    "OpenSource",
    "LeetCode",
    "Communication",
    "Teamwork",
    "Leadership",

    "EmployabilityStatus"
]

missing_columns = [
    c for c in required_columns
    if c not in feature_check.columns
]

print("Missing required columns:", missing_columns)

assert len(missing_columns) == 0
assert feature_check.shape == (15000, 29)
assert feature_check.isnull().sum().sum() == 0
assert feature_check["Student_ID"].nunique() == 5000
assert feature_check["event_timestamp"].nunique() == 3

print("TechnicalSkillScore range:",
      round(feature_check["TechnicalSkillScore"].min(), 2),
      "to",
      round(feature_check["TechnicalSkillScore"].max(), 2))

print("✓ Feature dataset is valid")


# ------------------------------------------------------------
# 3. FEAST ENTITY
# ------------------------------------------------------------

print("\n[3/8] FEAST ENTITY")

entity = store.get_entity("student")

print("Entity name:", entity.name)
print("Entity object:", entity)

assert entity.name == "student"

# We already defined the entity with Student_ID.
# Feast 0.65 does not expose join_keys as entity.join_keys.
print("Configured join key: Student_ID")

print("✓ Entity is registered correctly")


# ------------------------------------------------------------
# 4. FEAST FEATURE VIEW
# ------------------------------------------------------------

print("\n[4/8] FEAST FEATURE VIEW")

fv = store.get_feature_view("student_features")

print("FeatureView name:", fv.name)

registered_features = [
    feature.name
    for feature in fv.features
]

print("\nRegistered features:")

for feature_name in registered_features:
    print("  ✓", feature_name)

print("\nNumber of Feast features:",
      len(registered_features))

assert fv.name == "student_features"
assert "TechnicalSkillScore" in registered_features
assert len(registered_features) == 26

print("✓ FeatureView is registered correctly")


# ------------------------------------------------------------
# 5. HISTORICAL FEATURE RETRIEVAL
# ------------------------------------------------------------

print("\n[5/8] HISTORICAL FEATURE RETRIEVAL")

entity_df = feature_check[
    [
        "Student_ID",
        "event_timestamp",
        "EmployabilityStatus"
    ]
].copy()

feature_refs = [
    "student_features:CGPA",
    "student_features:Attendance",
    "student_features:Backlogs",
    "student_features:Year",

    "student_features:Python",
    "student_features:Java",
    "student_features:DSA",
    "student_features:SQL",
    "student_features:OOP",
    "student_features:DBMS",
    "student_features:OS",
    "student_features:CN",
    "student_features:Git",
    "student_features:Linux",
    "student_features:Docker",
    "student_features:Cloud",

    "student_features:TechnicalSkillScore",

    "student_features:Projects",
    "student_features:Internship",
    "student_features:Certifications",
    "student_features:Hackathons",
    "student_features:OpenSource",
    "student_features:LeetCode",

    "student_features:Communication",
    "student_features:Teamwork",
    "student_features:Leadership"
]

print("Requesting historical features...")

historical_job = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs
)

historical_df = historical_job.to_df()

print("Historical dataset shape:",
      historical_df.shape)

print("TechnicalSkillScore retrieved:",
      "TechnicalSkillScore" in historical_df.columns)

print("Missing values:",
      int(historical_df.isnull().sum().sum()))

assert historical_df.shape == (15000, 29)
assert "TechnicalSkillScore" in historical_df.columns

print("✓ Historical retrieval successful")


# ------------------------------------------------------------
# 6. MACHINE LEARNING MODEL
# ------------------------------------------------------------

print("\n[6/8] MACHINE LEARNING MODEL")

TARGET = "EmployabilityStatus"

X = historical_df.drop(
    columns=[
        "Student_ID",
        "event_timestamp",
        TARGET
    ]
)

y = historical_df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Model type:",
      type(feast_model).__name__)
print("Model expects:",
      feast_model.n_features_in_,
      "features")
print("Model classes:",
      feast_model.classes_)

assert X.shape[1] == 26
assert feast_model.n_features_in_ == 26

print("✓ Model is compatible")


# ------------------------------------------------------------
# 7. ONLINE FEATURE RETRIEVAL
# ------------------------------------------------------------

print("\n[7/8] ONLINE FEATURE RETRIEVAL")

student_id = 1

online_features = store.get_online_features(
    features=feature_refs,
    entity_rows=[
        {
            "Student_ID": student_id
        }
    ]
).to_dict()

print("Student ID:", student_id)

print("\nTechnicalSkillScore:",
      online_features.get("TechnicalSkillScore"))

assert "TechnicalSkillScore" in online_features

print("✓ Online feature retrieval successful")


# ------------------------------------------------------------
# 8. FINAL PREDICTION
# ------------------------------------------------------------

print("\n[8/8] FINAL PREDICTION")

model_feature_names = [
    "CGPA",
    "Attendance",
    "Backlogs",
    "Year",
    "Python",
    "Java",
    "DSA",
    "SQL",
    "OOP",
    "DBMS",
    "OS",
    "CN",
    "Git",
    "Linux",
    "Docker",
    "Cloud",
    "TechnicalSkillScore",
    "Projects",
    "Internship",
    "Certifications",
    "Hackathons",
    "OpenSource",
    "LeetCode",
    "Communication",
    "Teamwork",
    "Leadership"
]

X_online = pd.DataFrame({
    feature: online_features[feature]
    for feature in model_feature_names
})

prediction = feast_model.predict(X_online)[0]
probabilities = feast_model.predict_proba(X_online)[0]

print("Student ID:", student_id)
print("Predicted class:", int(prediction))

print("\nPrediction probabilities:")

for class_id, probability in zip(
    feast_model.classes_,
    probabilities
):
    print(
        f"  Class {int(class_id)}: "
        f"{probability:.2%}"
    )


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("              FEAST ASSIGNMENT VERIFICATION")
print("=" * 70)

print("✓ Original dataset")
print("✓ Feature engineering")
print("✓ Feast Entity")
print("✓ Feast FileSource")
print("✓ Feast FeatureView")
print("✓ feast apply / registration")
print("✓ Historical feature retrieval")
print("✓ ML model")
print("✓ Materialization / online store")
print("✓ Online feature retrieval")
print("✓ Final prediction")

print("\nTECHNICAL IMPLEMENTATION: COMPLETE")
print("=" * 70)

       CURRICULUM-INDUSTRY SKILL GAP — FEAST VERIFICATION

[1/8] ORIGINAL DATASET
Shape: (5000, 27)
Missing values: 0
Duplicate Student IDs: 0
✓ Original dataset is valid

[2/8] FEATURE DATASET
Shape: (15000, 29)
Missing required columns: []
TechnicalSkillScore range: 0.0 to 83.33
✓ Feature dataset is valid

[3/8] FEAST ENTITY
Entity name: student
Entity object: {
  "spec": {
    "name": "student",
    "valueType": "INT64",
    "description": "Student identified by Student_ID",
    "joinKey": "Student_ID"
  },
  "meta": {
    "createdTimestamp": "2026-08-17T06:23:24.530268Z",
    "lastUpdatedTimestamp": "2026-08-17T06:43:31.191745Z"
  }
}
Configured join key: Student_ID
✓ Entity is registered correctly

[4/8] FEAST FEATURE VIEW
FeatureView name: student_features

Registered features:
  ✓ CGPA
  ✓ Attendance
  ✓ Backlogs
  ✓ Year
  ✓ Python
  ✓ Java
  ✓ DSA
  ✓ SQL
  ✓ OOP
  ✓ DBMS
  ✓ OS
  ✓ CN
  ✓ Git
  ✓ Linux
  ✓ Docker
  ✓ Cloud
  ✓ TechnicalSkillScore
  ✓ Projects
  ✓ Internship
 